## Create and Destroy an Azure Kubernetes Service in Azure using Terraform

- An Azure Kubernetes Service (AKS) is a kubernetes cluster running in Azure.
  - Since an Azure Kubernetes Service (AKS) is an Azure resource, it must be placed in an Azure Resource Group.
  - Since an Azure Kubernetes Service (AKS) creates a Virtual Network, we also need to create a Network Watcher.
  - Since an Azure Kubernetes Service (AKS) needs to pull its images from a Registry, we also create an Azure Container Registry.
- This Terraform Project consists of the Terraform files listed below:

In [1]:
!dir *.tf
#!ls *.tf # use this on Linux/Mac

 Volume in drive C is Windows
 Volume Serial Number is 3C6C-8E33

 Directory of c:\Users\PAGA\projects\devops\workshop5\01_Azure_and_Terraform\04_kubernetes_cluster

01/28/2025  06:30               655 container-registry.tf
01/28/2025  06:41             1,661 kubernetes-cluster.tf
12/22/2024  12:29               778 network-watcher.tf
01/27/2025  17:45               321 providers.tf
01/27/2025  17:46               152 resource-group.tf
01/28/2025  06:41               328 variables.tf
               6 File(s)          3,895 bytes
               0 Dir(s)  50,574,262,272 bytes free


## Terraform providers

- We are using the same Terraform proviers as before (i.e. the `azurerm` provider for Azure).

In [2]:
!type providers.tf
#!cat providers.tf # use this on Linux/Mac

# Initialises Terraform providers and sets their version numbers.

terraform {
  required_providers {
    azurerm = {
      source  = "hashicorp/azurerm"
      version = "~> 4.14.0"
    }
  }

  required_version = ">= 1.10.3"
}

provider "azurerm" {
  features {}
  subscription_id = var.subscription_id
}


## Terraform variables

- We are using the same Terraform variables as before
  - But have added an additional variable with the name `kubernetes_version` and the value `1.30.6`.
  - This variable is used to set the Kubernetes version to use in the file `kubernetes-cluster.tf`.

**Note! Make sure you change the value for the variable `app_name` to something unique and set your `subscription_id`!**

In [3]:
!type variables.tf
#!cat variables.tf # use this on Linux/Mac

# Sets global variables for this Terraform project.

variable "subscription_id" {
  description = "The Azure subscription ID"
  type        = string
}

variable "app_name" {
  default = "flixtube2025g00"
}

variable "location" {
  default = "westeurope"
}

variable "kubernetes_version" {
  default = "1.30.6"
}


## Azure Resource Group

- We are using the same Azure Resoure Group as before.

In [4]:
!type resource-group.tf
#!cat resource-group.tf # use this on Linux/Mac

# Creates a resource group in your Azure account.

resource "azurerm_resource_group" "main" {
  name     = var.app_name
  location = var.location
}


## Azure Container Registry

- We are using the same Azure Container Registry as before.
- We need this to store our Microservices' Images.
- When creating Pods in the Azure Kubernetes Service (AKS) cluster, the Images will be pulled from this Azure Container Registry.

In [5]:
!type container-registry.tf
#!cat container-registry.tf # use this on Linux/Mac

# Creates a container registry in Azure (for Docker images).

resource "azurerm_container_registry" "main" {
  name                = var.app_name
  resource_group_name = azurerm_resource_group.main.name
  location            = var.location
  admin_enabled       = true
  sku                 = "Basic"
}

output "AZURE_CONTAINER_REGISTRY_HOSTNAME" {
  value = azurerm_container_registry.main.login_server
}

output "AZURE_CONTAINER_REGISTRY_USERNAME" {
  value = azurerm_container_registry.main.admin_username
}

output "AZURE_CONTAINER_REGISTRY_PASSWORD" {
  value = azurerm_container_registry.main.admin_password
  sensitive = true
}


## Let's view the contents of the file `kubernetes-cluster.tf`

- Here we are defining an Azure Kubernetes Service (AKS), i.e. a Kubernetes cluster on Azure
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_kubernetes_cluster`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `kubernetes_cluster` is the name of the Azure resource (i.e. an Azure Kubernetes Service defined in the `azurerm` provider/plugin).
  - The first Argument sets the Azure Kubernetes Service's name
    - `name` is the argument's name
    - Its value is retrieved from the Terraform variable `app_name` (defined in the file `variables.tf`).
  - The second Argument sets the Azure Kubernetes Service's Location
    - `location` is the argument's name
    - Its value is retrieved from the Terraform variable `location` (defined in the file `variables.tf`).
  - The third Argument sets the Azure Resource Group in which the Azure Kubernetes Service will be created
    - `resource_group_name` is the argument's name
    - Its value is retrieved from the Terraform Expression `azurerm_resource_group.main.name`.
      - The `azurerm_resource_group.main` Block is defined in `resource-group.tf` as `resource "azurerm_resource_group" "main"`.
      - In this Block, there is an Argument with a name of `name` who's value is defined as `var.app_name`.
      - This is the value that is assigned to `resource_group_name`.
  - The fourth Argument sets the DNS Prefix for the Azure Kubernetes Service
    - `dns_prefix` is the argument's name
    - Its value is retrieved from the Terraform variable `app_name` (defined in the file `variables.tf`).
    - This parameter specifies the prefix to use for hostnames that are created for the DNS service in the cluster.
      - If not specified, a hostname is generated using a combination of the cluster's name and the resource group's name.
  - The fifth Argument sets version of Kubernetes to use for the cluster.
    - `kubernetes_version` is the argument's name
    - Its value is retrieved from the Terraform variable `kubernetes_version` (defined in the file `variables.tf`).
- The Block also contains two nested Blocks.
  - The first nested Block type is `default_node_pool`
    - An AKS cluster creates a Virtual Machine for each Node, and uses a "pool" for its Nodes, so that they can be scaled when required.
    - The first Argument sets the node pool's name
      - `name` is the argument's name
      - Its value is set to `"default"`.
    - The second Argument sets the initial number of Nodes in the pool
      - `node_count` is the argument's name
      - Its value is set to `1`.
    - The third Argument sets the size of the Virtual Machine used for each Node in the pool
      - `vm_size` is the argument's name
      - Its value is set to `"Standard_D2s_V3"`.
      - For more information about Virtual Machine Sizes, see:
        - https://learn.microsoft.com/en-us/azure/virtual-machines/sizes/general-purpose/dsv3-series
  - The second nested Block type is `identity`
    - A AKS cluster needs an identity (service principle) which can be created manually or automatically.
    - The first and only Argument sets the AKS cluster's identity
      - `type` is the argument's name
      - Its value is set to `"SystemAssigned"`, which means Azure will create an identity (service principle) automatically.
      - For more information about Identities, see:
        - https://learn.microsoft.com/en-us/entra/identity/managed-identities-azure-resources/overview
- Lastly, we are defining an Azure Role Assignment, which automatically gives the AKS cluster access to the Container Registry.
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_role_assignment`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `role_assignment` is the name of the Azure resource (i.e. an Azure Role Assignment defined in the `azurerm` provider/plugin).
  - The first Argument sets the identity of the "principle" (object) that needs access to the Container Registry (a kubelet in the cluster)
    - `principal_id` is the argument's name
    - Its value is retrieved from the AKS resource `azurerm_kubernetes_cluster.main` we defined above, where
      - `kubelet_identity[0]` accesses the first item in this array property, and `object_id` is the identity of this kubelet.
  - The second Argument defines what type of access (authorization) the kubelet needs, which is to pull images from the Container Registry
    - `role_definition_name` is the argument's name
    - Its value is set to `"AcrPull"`, i.e. "Azure Container Registry Pull" which means the Kubelet is allowed to pull images.
  - The third Argument sets the scope (the identity of the object we are granting access to), which is the Azure Container Registry's identity
    - `scope` is the argument's name
    - Its value is set to `azurerm_container_registry.main.id`, where
      - `azurerm_container_registry.main` is the Azure Container Registry resource defined in the file `container-registry.tf`.
      - `id` is the property in that resource that contains the Azure Container Registry's identity.
  - The fourth Argument skips checking the kubelet's service principle (identity) in Azure Active Directory
    - `skip_service_principal_aad_check` is the argument's name.
    - Its value is set to `true`.
    - For more information see: https://registry.terraform.io/providers/hashicorp/azurerm/latest/docs/resources/role_assignment

In [6]:
!type kubernetes-cluster.tf
#!cat kubernetes-cluster.tf # use this on Linux/Mac

# Creates a managed Kubernetes cluster on Azure.
# Note!
# - Resource "azurerm_resource_group.main" with a property "name" is defined in the file "resource-group.tf".
# - The value for "resource_group_name" below is set using property "name" in resource "azurerm_resource_group.main":
#   - resource_group_name = azurerm_resource_group.main.name
# - "name", "location" and "kubernetes_version" below are set from Terraform variables defined in the file "variables.tf".

resource "azurerm_kubernetes_cluster" "main" {
  name                = var.app_name
  location            = var.location
  resource_group_name = azurerm_resource_group.main.name
  dns_prefix          = var.app_name
  kubernetes_version  = var.kubernetes_version

  default_node_pool {
    name       = "default"
    node_count = 1
    vm_size    = "Standard_D2S_V3"
  }

  # Instead of creating a service principle have the system figure this out.

  identity {
    type = "SystemAssigned"
  }
}

# Attach the Container Registry t

## Let's view the contents of the file `network-watcher.tf`

- Microsoft requires every Virtual Network in Azure to have a Network Watcher.
  - Since an Azure Kubernetes Service (AKS) creates a Virtual Network, we also need a Network Watcher.
  - A Network Watcher is created automatically for a Virtual Network, but we are creating it manually here.
    - The only readson for creating it manually here, is so that a `terraform destroy` will delete it for us.
- Here we are defining two resources `azurerm_resource_group` and `azurerm_network_watcher`
  - `azurerm_resource_group` is a new Resource Group for the Network Watcher.
  - `azurerm_network_watcher` is the Network Watcher (which we place in the new Resource Group).
- For the Resource Group
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_resource_group"`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `resource_group` is the name of the Azure resource (i.e. an Azure Resource Group defined in the `azurerm` provider/plugin).
  - The second Block Label is `networkwatcher"` which is used to uniquely identity this resource in the Terraform files.
  - The first Argument sets the Azure Resource Groups's name
    - `name` is the argument's name
    - Its value is set to `NetworkWatcherRG`.
  - The second Argument sets the Azure Resource Group's Location
    - `location` is the argument's name
    - Its value is retrieved from the Terraform variable `location` (defined in the file `variables.tf`).
- For the Network Watcher
  - The Block Type is `resource`.
  - The first Block Label is `azurerm_network_watcher"`
    - `azurerm` is the name of the provider (i.e. the provider for Azure defined in the file `providers.tf`).
    - `network_watcher` is the name of the Azure resource (i.e. an Azure Network Watcher defined in the `azurerm` provider/plugin).
  - The second Block Label is `networkwatcher"` which is used to uniquely identity this resource in the Terraform files.
  - The first Argument sets the Azure Network Watcher's name
    - `name` is the argument's name
    - Its value is set to `NetworkWatcher_westeurope`.
  - The second Argument sets the Azure Network Wacther's Location
    - `location` is the argument's name
    - Its value is retrieved from the Terraform variable `location` (defined in the file `variables.tf`).
  - The third Argument places the Network Watcher in the Azure Resource Group we defined above
    - `resource_group_name` is the argument's name
    - Its value is retrieved from `azurerm_resource_group.networkwatcher.name`, where
      - `azurerm_resource_group.networkwatcher` refers to the Azure Resource Group we defined above.
      - `name` is the name property in the Azure Resource Group, which returns the name of the Resource Group.

In [7]:
!type network-watcher.tf
#!cat network-watcher.tf # use this on Linux/Mac

# Creates a Network Watcher on Azure.
# Note!
# - We create a "networkwatcher" (in its own Resource Group).
#   - This is required when a virtual network is created in Azure.
#     - An Azure Kubernetes Service (AKS) will create a virtual network.
#   - This is automatically created by Azure, but we explicitly create it here.
#     - The only reason we do this explicitly is so Terraform Destroy will automatically delete it for us.

resource "azurerm_resource_group" "networkwatcher" {
  name     = "NetworkWatcherRG"
  location = var.location
}

resource "azurerm_network_watcher" "networkwatcher" {
  name                = "NetworkWatcher_westeurope"
  location            = var.location
  resource_group_name = azurerm_resource_group.networkwatcher.name
}


## Initialize Terraform

- We initialize the Terraform Project as before.

In [8]:
#rm -rf .terraform rm .terraform.lock.hcl terraform.tfstate terraform.tfstate.backup
!terraform init

Initializing the backend...
Initializing provider plugins...
- Finding hashicorp/azurerm versions matching "~> 4.14.0"...
- Installing hashicorp/azurerm v4.14.0...
- Installed hashicorp/azurerm v4.14.0 (signed by HashiCorp)
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.


## Terraform Apply

- We apply the Terraform Project as before.

**Note!**

- Once again it's better to run this command in a separate terminal.
  - Open a new terminal.
  - Make sure you are in the folder `workshop5/01_Azure_and_Terraform/04_kubernetes_cluster`
  - Execute the command `terraform apply -auto-approve`

The output should look something like the below ...

```bash
Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azurerm_container_registry.main will be created
  + resource "azurerm_container_registry" "main" {
      + admin_enabled                 = true
      + admin_password                = (sensitive value)
      + admin_username                = (known after apply)
      + encryption                    = (known after apply)
      + export_policy_enabled         = true
      + id                            = (known after apply)
      + location                      = "westeurope"
      + login_server                  = (known after apply)
      + name                          = "flixtube2025g00"
      + network_rule_bypass_option    = "AzureServices"
      + network_rule_set              = (known after apply)
      + public_network_access_enabled = true
      + resource_group_name           = "flixtube2025g00"
      + sku                           = "Basic"
      + trust_policy_enabled          = false
      + zone_redundancy_enabled       = false
    }

  # azurerm_kubernetes_cluster.main will be created
  + resource "azurerm_kubernetes_cluster" "main" {
      + current_kubernetes_version          = (known after apply)
      + dns_prefix                          = "flixtube2025g00"
      + fqdn                                = (known after apply)
      + http_application_routing_zone_name  = (known after apply)
      + id                                  = (known after apply)
      + kube_admin_config                   = (sensitive value)
      + kube_admin_config_raw               = (sensitive value)
      + kube_config                         = (sensitive value)
      + kube_config_raw                     = (sensitive value)
      + kubernetes_version                  = "1.30.6"
      + location                            = "westeurope"
      + name                                = "flixtube2025g00"
      + node_os_upgrade_channel             = "NodeImage"
      + node_resource_group                 = (known after apply)
      + node_resource_group_id              = (known after apply)
      + oidc_issuer_url                     = (known after apply)
      + portal_fqdn                         = (known after apply)
      + private_cluster_enabled             = false
      + private_cluster_public_fqdn_enabled = false
      + private_dns_zone_id                 = (known after apply)
      + private_fqdn                        = (known after apply)
      + resource_group_name                 = "flixtube2025g00"
      + role_based_access_control_enabled   = true
      + run_command_enabled                 = true
      + sku_tier                            = "Free"
      + support_plan                        = "KubernetesOfficial"
      + workload_identity_enabled           = false

      + auto_scaler_profile (known after apply)

      + default_node_pool {
          + kubelet_disk_type    = (known after apply)
          + max_pods             = (known after apply)
          + name                 = "default"
          + node_count           = 1
          + node_labels          = (known after apply)
          + orchestrator_version = (known after apply)
          + os_disk_size_gb      = (known after apply)
          + os_disk_type         = "Managed"
          + os_sku               = (known after apply)
          + scale_down_mode      = "Delete"
          + type                 = "VirtualMachineScaleSets"
          + ultra_ssd_enabled    = false
          + vm_size              = "Standard_D2S_V3"
          + workload_runtime     = (known after apply)
        }

      + identity {
          + principal_id = (known after apply)
          + tenant_id    = (known after apply)
          + type         = "SystemAssigned"
        }

      + kubelet_identity (known after apply)

      + network_profile (known after apply)

      + windows_profile (known after apply)
    }

  # azurerm_network_watcher.networkwatcher will be created
  + resource "azurerm_network_watcher" "networkwatcher" {
      + id                  = (known after apply)
      + location            = "westeurope"
      + name                = "NetworkWatcher_westeurope"
      + resource_group_name = "NetworkWatcherRG"
    }

  # azurerm_resource_group.main will be created
  + resource "azurerm_resource_group" "main" {
      + id       = (known after apply)
      + location = "westeurope"
      + name     = "flixtube2025g00"
    }

  # azurerm_resource_group.networkwatcher will be created
  + resource "azurerm_resource_group" "networkwatcher" {
      + id       = (known after apply)
      + location = "westeurope"
      + name     = "NetworkWatcherRG"
    }

  # azurerm_role_assignment.main will be created
  + resource "azurerm_role_assignment" "main" {
      + condition_version                = (known after apply)
      + id                               = (known after apply)
      + name                             = (known after apply)
      + principal_id                     = (known after apply)
      + principal_type                   = (known after apply)
      + role_definition_id               = (known after apply)
      + role_definition_name             = "AcrPull"
      + scope                            = (known after apply)
      + skip_service_principal_aad_check = true
    }

Plan: 6 to add, 0 to change, 0 to destroy.

Changes to Outputs:
  + AZURE_CONTAINER_REGISTRY_HOSTNAME = (known after apply)
  + AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value)
  + AZURE_CONTAINER_REGISTRY_USERNAME = (known after apply)
azurerm_resource_group.main: Creating...
azurerm_resource_group.networkwatcher: Creating...
azurerm_resource_group.main: Still creating... [10s elapsed]
azurerm_resource_group.networkwatcher: Still creating... [10s elapsed]
azurerm_resource_group.networkwatcher: Creation complete after 11s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG]
azurerm_network_watcher.networkwatcher: Creating...
azurerm_resource_group.main: Creation complete after 12s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Creating...
azurerm_kubernetes_cluster.main: Creating...
azurerm_network_watcher.networkwatcher: Creation complete after 2s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope]
azurerm_container_registry.main: Still creating... [10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [10s elapsed]
azurerm_container_registry.main: Still creating... [20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [20s elapsed]
azurerm_container_registry.main: Creation complete after 25s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_kubernetes_cluster.main: Still creating... [30s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [40s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [50s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m0s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m30s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m40s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m50s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m0s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m30s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m40s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m50s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [3m0s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [3m10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [3m20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [3m30s elapsed]
azurerm_kubernetes_cluster.main: Creation complete after 3m33s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00]
azurerm_role_assignment.main: Creating...
azurerm_role_assignment.main: Still creating... [10s elapsed]
azurerm_role_assignment.main: Still creating... [20s elapsed]
azurerm_role_assignment.main: Creation complete after 22s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/a3abce92-e84a-94c5-c9dc-a5b2074e1fde]

Apply complete! Resources: 6 added, 0 changed, 0 destroyed.

Outputs:

AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io"
AZURE_CONTAINER_REGISTRY_PASSWORD = <sensitive>
AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00"
```

In [ ]:
# !terraform apply -auto-approve

## List Azure Resource Groups

- The Azure CLI command `az group list -o table` lists all Resource Groups in Azure.
- We see that the two Resource Groups defined in the Terraform project has been created.
- We also see an additional Resource Group `MC_flixtube2025g00_flixtube2025g00_westeurope` that is automatically created by Azure.
  - This Resource Group isn't included in the Terraform Project, but will automatically be destroyed by `terraform destroy`.

In [10]:
!az group list -o table

Name                                           Location    Status
---------------------------------------------  ----------  ---------
flixtube2025g00                                westeurope  Succeeded
NetworkWatcherRG                               westeurope  Succeeded
MC_flixtube2025g00_flixtube2025g00_westeurope  westeurope  Succeeded


## List Resources in Resource Group `flixtube2025g00`

- The Azure CLI command `az resource list -n flixtube2025g00 -o table` lists resources in Resource Group `flixtube2025g00`.
- We see that the Resource Group contains the Azure Kubernetes Service and the Azure Container Registry.

In [11]:
!az resource list -n flixtube2025g00 -o table

Name             ResourceGroup    Location    Type                                        Status
---------------  ---------------  ----------  ------------------------------------------  --------
flixtube2025g00  flixtube2025g00  westeurope  Microsoft.ContainerService/managedClusters
flixtube2025g00  flixtube2025g00  westeurope  Microsoft.ContainerRegistry/registries


## List Azure Container Registries

- The Azure CLI command `az acr list -o table` lists all Container Registries in Azure.
- We see that the Container Registry defined in the Terraform project has been created.

In [ ]:
!az acr list -o table

## List Repositories in Container Registry `flixtube2025g00 `

- The Azure CLI command `az acr repository list -n flixtube2025g00 --top 10 -o table` lists Repositories in a Azure Container Registry `flixtube2025g00`.
  - The `-n` option is manditory and specifies the `NAME` of the Container Registry.
  - The `--top 10` limits the list to the first 10 Repositories (remove to see all Repositories).
- We see that Container Registry `flixtube2025g00` doesn't contain any Repositories.

In [13]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

## Show Information about Azure Container Registry `flixtube2025g00`

- The Azure CLI command `az acr show -n flixtube2025g00 -o table` shows information about Container Registry `flixtube2025g00`.
- It shows the Container Registry's `LOGIN SERVER` which is the URL to your Container Registry on Azure.
  - **This is the URL you would use to upload Docker Images to the Azure Container Registry.**

In [ ]:
!az acr show -n flixtube2025g00 -o table

# Let's store the LOGIN SERVER in a Python variable so we can use it later in this notebook
CONTAINER_REGISTRY_LOGIN_SERVER=!az acr show -n flixtube2025g00 --query loginServer -o tsv
CONTAINER_REGISTRY_LOGIN_SERVER=CONTAINER_REGISTRY_LOGIN_SERVER[0]

## Show Credentials for Azure Container Registry `flixtube2025g00`

- The Azure CLI command `az acr credential show -n flixtube2025g00 -o table` shows credentials about Container Registry `flixtube2025g00`.
- It shows the Container Registry's `USERNAME` and  `PASSWORD` to use to authenticate with your Azure Container Registry.
  - **This is the USERNAME and PASSWORD you would use to login to Docker to upload Images to the Azure Container Registry.**

In [ ]:
!az acr credential show -n flixtube2025g00 -o table

# Let's store the USERNAME and PASSWORD in Python variables so we can use them later in this notebook
CONTAINER_REGISTRY_USERNAME=!az acr credential show -n flixtube2025g00 --query username -o tsv
CONTAINER_REGISTRY_USERNAME=CONTAINER_REGISTRY_USERNAME[0]
CONTAINER_REGISTRY_PASSWORD=!az acr credential show -n flixtube2025g00 --query passwords[0].value -o tsv
CONTAINER_REGISTRY_PASSWORD=CONTAINER_REGISTRY_PASSWORD[0]

## Login to Azure Container Registry via Docker

- Replace the variables below with your `LOGIN_SERVER`, `USERNAME` and `PASSWORD`.

In [16]:
!docker login $CONTAINER_REGISTRY_LOGIN_SERVER -u $CONTAINER_REGISTRY_USERNAME -p $CONTAINER_REGISTRY_PASSWORD

# In Ubuntu with environment variables CONTAINER_REGISTRY_LOGIN_SERVER, CONTAINER_REGISTRY_USERNAME and CONTAINER_REGISTRY_USERNAME
#!echo $PASSWORD | docker login $LOGIN_SERVER -u $USERNAME --password-stdin  > /dev/null 2>&1

Login Succeeded


WARNING! Using --password via the CLI is insecure. Use --password-stdin.


## Let's view the code in `Flixtube.VideoStreaming`

- The file `Flixtube.VideoStreaming/Flixtube.VideoStreaming/Program.cs` is listed below.
  - At the top of the file, it reads in a couple of environment variables.
    - `FLIXTUBE_VIDEO_STREAMING_PORT` is the port the microservice will listen on.
    - `FLIXTUBE_STORAGE_FOLDER_NAME` is the name of the filesystem folder it will serve video files from.
  - These environment variables are stored in the `IConfiguration` instance, after discarding their `FLIXTUBE_` prefix.
  - Then it configures the service container and the HTTP Request/Response pipeline as usual for an ASP.NET Web API project.
  - Lastly, it starts the microservice listening on port `FLIXTUBE_VIDEO_STREAMING_PORT`.

In [17]:
!type Flixtube.VideoStreaming\Flixtube.VideoStreaming\Program.cs
#!cat Flixtube.VideoStreaming/Flixtube.VideoStreaming/Program.cs # use this on Linux/Mac

using Scalar.AspNetCore;

var builder = WebApplication.CreateBuilder(args);

// Make sure the necessary environment variables are available.

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_VIDEO_STREAMING_PORT"))) {
    throw new Exception("Please specify the port number for Flixtube.VideoStreaming with the environment variable FLIXTUBE_VIDEO_STREAMING_PORT.");
}

if (string.IsNullOrEmpty(Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_FOLDER_NAME"))) {
    throw new Exception("Please specify the Filesystem folder name for Flixtube.VideoStreaming with the subkey FLIXTUBE_STORAGE_FOLDER_NAME.");
}

// Get necessary environment variables
// Note that we only need to get settings here if there are need before builder.Build()
// int VIDEO_STREAMING_PORT = int.Parse(Environment.GetEnvironmentVariable("FLIXTUBE_VIDEO_STREAMING_PORT") ?? "80");
// string STORAGE_FOLDER_NAME = Environment.GetEnvironmentVariable("FLIXTUBE_STORAGE_FOLDER_NAME") ?? string.Empty;

// On

- The file `Flixtube.VideoStreaming/Flixtube.VideoStreaming/Controllers/VideoStreamingController.cs` is listed below.
  - At the top of the file, it reads in an environment variable stored in the dependency-injected `IConfiguration` instance.
    - `STORAGE_FOLDER_NAME` is the name of the filesystem folder the microservice will serve video files from.
  - The file also contains a number of HTTP endpoints, where one of these is the `/{id}` route that streams a video.
    - `{id}` is the name of the video file in the microservice's filesystem.
    - `STORAGE_FOLDER_NAME` is the name of the folder the microservice serves video files from.
    - `STORAGE_FOLDER_NAME/id` is the path to the video file in the microservice's filesystem.
- So the microservice will simply stream video files stored in its filesystem when `HTTP GET /{id}` is called. 

In [18]:
!type Flixtube.VideoStreaming\Flixtube.VideoStreaming\Controllers\VideoStreamingController.cs
#!cat Flixtube.VideoStreaming/Flixtube.VideoStreaming/Controllers/VideoStreamingController.cs # use this on Linux/Mac

using System.Net;
using Microsoft.AspNetCore.Mvc;

namespace Flixtube.VideoStreaming.Controllers;

[ApiController]
[Route("/")]
public class VideoStreamingController : ControllerBase
{
    private readonly ILogger<VideoStreamingController> _logger;
    private readonly IConfiguration _config;
    private readonly string STORAGE_FOLDER_NAME;

    public VideoStreamingController(ILogger<VideoStreamingController> logger, IConfiguration config)
    {
        _logger = logger;
        _config = config;

        STORAGE_FOLDER_NAME = _config.GetValue<string>("STORAGE_FOLDER_NAME")!;

        _logger.LogInformation("VideoStreamingController() called.");
    }

    // Health check.
    [HttpGet("/health")]
    public async Task<IActionResult> Health()
    {
        await Task.Delay(0);
        return Ok();
    }

    // Stream video from the Filesystem.
    [HttpGet("{id}")]
    public async Task StreamVideo(string id)
    {
        // _logger.LogInformation($"StreamVideo() called.");
        

## Let's look at the Docker file for `Flixtube.VideoStreaming`

- The Dockerfile `Flixtube.VideoStreaming\Dockerfile` is listed below.
  - It base image has `.net sdk 9.0` pre-installed.
  - It sets the `WORKDIR` to `/src` and copies all code for the microservice to it.
  - It also copies the `videos` folder to `/src`.
    - This folder contains one sample video file `SampleVideo_1280x720_1mb-mp4`.
  - Then it restores (installs) all NuGet packages.
  - Finally, it starts the microservice, by issuing the command below when the container starts.
    - `dotnet watch run --no-launch-profile --project Flixtube.VideoStreaming.csproj`

In [19]:
!type Flixtube.VideoStreaming\Dockerfile
#!cat Flixtube.VideoStreaming/Dockerfile # use this on Linux/Mac

FROM mcr.microsoft.com/dotnet/sdk:9.0
WORKDIR /src
EXPOSE 80
COPY ./Flixtube.VideoStreaming ./
COPY ./videos ./videos
RUN ["dotnet","restore"]
CMD ["dotnet","watch","run","--no-launch-profile","--project","Flixtube.VideoStreaming.csproj"]


## Build and Push a Docker Image to Azure Container Registry

- Here we are building an image of the ASP.NET Web API application (microservice).
- We are tagging the image as `flixtube2025g00.azurecr.io/video-streaming:1,`where:
  - `flixtube2025g00.azurecr.io` is the URL (LOGIN SERVER) to our Container Registry.
  - `video-streaming` is the name of our image (repository).
  - `1` is the version of the image (tag).
- Then the image is pushed to the Azure Container Registry.
- Finally, the local image is removed from the host computer.

In [20]:
# Build Docker image with Nodejs Application
!docker build -q -t {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1 -f ./Flixtube.VideoStreaming/Dockerfile ./Flixtube.VideoStreaming

# Push Docker Image to Azure Container Registry
!docker push {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1

# Clean up
!docker rmi {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1
!docker images {CONTAINER_REGISTRY_LOGIN_SERVER}/video-streaming:1

sha256:4027094e62881b5332b674e1b6b47f1d9f5545d78812634a3a63bf2c32bd3e7f
The push refers to repository [flixtube2025g00.azurecr.io/video-streaming]
45faf544d0e6: Preparing
dcffe1b8a5c5: Preparing
515210e4a4a7: Preparing
cb5d5d0673d7: Preparing
1bb564ecf252: Preparing
7e9d6a3f8c92: Preparing
2e7a3a1e4448: Preparing
064dc71a1978: Preparing
c9aaa778af8e: Preparing
5479f1788e98: Preparing
d54764d9a8e7: Preparing
3b245d6409b1: Preparing
f5fe472da253: Preparing
7e9d6a3f8c92: Waiting
2e7a3a1e4448: Waiting
064dc71a1978: Waiting
c9aaa778af8e: Waiting
5479f1788e98: Waiting
d54764d9a8e7: Waiting
3b245d6409b1: Waiting
f5fe472da253: Waiting
cb5d5d0673d7: Pushed
dcffe1b8a5c5: Pushed
45faf544d0e6: Pushed
515210e4a4a7: Pushed
1bb564ecf252: Pushed
c9aaa778af8e: Pushed
d54764d9a8e7: Pushed
064dc71a1978: Pushed
2e7a3a1e4448: Pushed
5479f1788e98: Pushed
3b245d6409b1: Pushed
f5fe472da253: Pushed
7e9d6a3f8c92: Pushed
1: digest: sha256:95e0a8492c75f330ee827f0e798df1e9c9962a1bce58cf4b4d88b90586325387 size: 305

## Logout from the Azure Container Registry via Docker

In [21]:
!docker logout $CONTAINER_REGISTRY_LOGIN_SERVER

Removing login credentials for flixtube2025g00.azurecr.io


## List Repositories in Container Registry `flixtube2025g00 `

- We see that the Repository `video-streaming` has been created in Container Registry `flixtube2025g00`.

In [22]:
!az acr repository list -n flixtube2025g00 --top 10 -o table

Result
---------------
video-streaming


## Show Information about Repository `video-streaming`

- The Azure CLI command `az acr repository show -n flixtube2025g00 --repository video-streaming -o table`
  - Shows information about Repository `video-streaming` in Container Registry `flixtube2025g00`.
    - It contains images named `video-streaming` (ImageName).
    - It has a tag count of `1` (TagCount), i.e. currently there is only one tag for the `video-streaming` image.

In [23]:
!az acr repository show -n flixtube2025g00 --repository video-streaming -o table

CreatedTime                   ImageName        LastUpdateTime                ManifestCount    Registry                    TagCount
----------------------------  ---------------  ----------------------------  ---------------  --------------------------  ----------
2025-01-28T06:15:22.3940014Z  video-streaming  2025-01-28T06:15:22.5152663Z  1                flixtube2025g00.azurecr.io  1


## List Tags in Repository `video-streaming`

- The Azure CLI command `az acr repository show-tags -n flixtube2025g00 --repository video-streaming --top 10 -o table`:
  - Lists the tags in Repository `video-streaming` in Container Registry `flixtube2025g00`.
    - Currently there is only one tag.
    - The tag has the value `1`.

In [24]:
!az acr repository show-tags -n flixtube2025g00 --repository video-streaming --top 10 -o table

Result
--------
1


## Show Information about Image `video-streaming:1`

- The Azure CLI command `az acr repository show -n flixtube2025g00 --image video-streaming:1 -o table`:
  - Shows information about image `video-streaming:1` in Container Registry `flixtube2025g00`.
    - The information includes the Digest for the image.

In [25]:
!az acr repository show -n flixtube2025g00 --image video-streaming:1 -o table

CreatedTime                   Digest                                                                   LastUpdateTime                Name    Signed
----------------------------  -----------------------------------------------------------------------  ----------------------------  ------  --------
2025-01-28T06:15:22.5538227Z  sha256:95e0a8492c75f330ee827f0e798df1e9c9962a1bce58cf4b4d88b90586325387  2025-01-28T06:15:22.5538227Z  1       False


## Show Azure Container Registry Usage

- The command `az acr show-usage -n flixtube2025g00 -o table` shows the usage of Container Registry `flixtube2025g00`.
  - We can see the maximum number of allowed bytes in the `LIMIT` column (first row).
  - We can see the current number of bytes in the `CURRENT VALUE` column (first row).

In [26]:
!az acr show-usage -n flixtube2025g00 -o table

NAME       LIMIT        CURRENT VALUE    UNIT
---------  -----------  ---------------  ------
Size       10737418240  318026174        Bytes
Webhooks   2            0                Count
ScopeMaps  100          0                Count
Tokens     100          0                Count


## List Azure Kubernetes Services

- The Azure CLI command `az aks list -o table` lists all Azure Kubernetes Services in Azure.
- We see that the Azure Kubernetes Service (AKS) defined in the Terraform project has been created.

In [27]:
!az aks list -o table

Name             Location    ResourceGroup    KubernetesVersion    CurrentKubernetesVersion    ProvisioningState    Fqdn
---------------  ----------  ---------------  -------------------  --------------------------  -------------------  -------------------------------------------------
flixtube2025g00  westeurope  flixtube2025g00  1.30.6               1.30.6                      Succeeded            flixtube2025g00-ed2io0ms.hcp.westeurope.azmk8s.io


## Add Azure Kubernetes Cluster Info. to Local Kubectl Config File

- The Azure CLI command `az aks get-credentials --name flixtube2025g00 --resource-group flixtube2025g00` will:
  - Get the credentials from the Azure Kubernetes Service `flixtube2025g00` in Resource Group `flixtube2025g00`.
  - Add the credentials to the 'kubectl' CLI tool's `config` file on the host machine.
    - The 'kubectl' CLI tool's `config` file is located at  `~/.kube/config` on Linux/macOs.
    - The 'kubectl' CLI tool's `config` file is located at  `%USERPROFILE%\.kube\config` on Windows.

In [28]:
!az aks get-credentials --name flixtube2025g00 --resource-group flixtube2025g00

# The command below attaches the Azure Kubernetes Cluster to the Azure Container Registry so that
# the Azure Kubernetes Cluster can pull images from the Azure Container Registry, but we don't
# have to do this explicitly here, since we do it in the Terraform file "kubernetes-cluster.tf".
#az aks update -n flixtube2025g00 -g flixtube2025g00 --attach-acr flixtube2025g00 -o table

## Ensure the Kubectl Context is set to the Azure Kubernetes Cluster

- The kubectl CLI command:
  - `kubectl config get-contexts` lists all contexts in kubectl's config file.
  - `kubectl config use-context [ContextName]` sets the current context to `[ContextName]` in kubectl's config file.
  - `kubectl config current-context` returns the current context set in kubectl's config file.
  - `kubectl config view` lists the contexts of kubectl's config file.
- Here we just want to make sure kubectl's current context is set to the Azure Kubernetes Service's context.
  - We see that the current context is `flixtube2025g00` (or whatever you set the `app_name` Terraform variable to).

In [ ]:
#!kubectl config get-contexts
#!kubectl config use-context flixtube2025g00
!kubectl config current-context
#!kubectl config view

flixtube2025g00


## List Deployments, Pods and Services

- The kubectl command:
  - `kubectl get deployments -o wide` lists all Deployments in the kubernetes cluster.
  - `kubectl get pods -o wide` lists all Pods in the kubernetes cluster.
  - `kubectl get services -o wide` lists all Services in the kubernetes cluster.
- Notice we have:
  - Not Deployments or Pods in the Azure Kubernetes Cluster.
  - One ClusterIP Service in the Azure Kubernetes Cluster already defined before deploying any resources ourselves.

In [30]:
!kubectl get deployments -o wide
!kubectl get pods -o wide
!kubectl get services -o wide

No resources found in default namespace.
No resources found in default namespace.


NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE   SELECTOR
kubernetes   ClusterIP   10.0.0.1     <none>        443/TCP   14m   <none>


## Lets' look at the code in the file `manifests/deployment.yaml`

- We see two kubenetes resources defined in the file:
  - `kind: Deployment` and `kind: Service`.
  - Notice how you can define multiple resources in a YAML file by separating them with three dashes `---`.
- Deployment:
  - The Deployment's `name` is `video-streaming`, sets `replicas` to `1`, and its `matchLabels` contains `app: video-streaming`.
  - The Pod template's `labels` contains `app: video-streaming` and its settings under `containers` are defined as below:
    - The container's `name` is `video-streaming`, and it pulls the `image` called `flixtube2025g00.azurecr.io/video-streaming:1`
      - `flixtube2025g00.azurecr.io` is the URL to the Azure Container Registry.
      - `video-streaming:1` is the name of the image, including the image's version (tag).
    - The container sets one environment variable named `FLIXTUBE_VIDEO_STREAMING_PORT` with the value `3000`.
    - The container sets one environment variable named `FLIXTUBE_STORAGE_FOLDER_NAME` with the value `videos`.
    - The container also `requests` `256m` of the node's `cpu` and `256Mi` of the node's `memory.`
    - The container also `limits` the use to `512m` of the node's `cpu` and `512Mi` o the node's `memory.`
- Service:
  - The Service's `name` is `video-streaming`, sets it's `type` to `LoadBalancer`, and its `matchLabels` contains `app: video-streaming`.
  - The Service is listening on `port` `80` and is redirecting traffic to `targetPort` `3000`.

In [45]:
!type manifests\deployment.yaml
#!cat manifests/deployment.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: video-streaming
spec:
  replicas: 1
  selector:
    matchLabels:
      app: video-streaming
  template:
    metadata:
      labels:
        app: video-streaming
    spec:
      containers: 
      - name: video-streaming
        image: flixtube2025g00.azurecr.io/video-streaming:1
        imagePullPolicy: IfNotPresent
        env:
        - name: FLIXTUBE_VIDEO_STREAMING_PORT
          value: "3000"
        - name: FLIXTUBE_STORAGE_FOLDER_NAME
          value: "videos"  
        resources:
          requests:
            cpu: 256m
            memory: 256Mi
          limits:
            cpu: 512m
            memory: 512Mi
---
apiVersion: v1
kind: Service
metadata:
  name: video-streaming
spec:
  selector:
    app: video-streaming
  type: LoadBalancer
  ports:
    - protocol: TCP
      port: 80
      targetPort: 3000


## Apply Deployment to the Azure Kubernets Cluster

- The kubectl command `kubectl apply -f manifests/deployment.yaml` will:
  -  Apply the resource definitions in the file `manifests/deployment.yaml` to the kubernetes cluster.

In [46]:
!kubectl apply -f manifests/deployment.yaml

deployment.apps/video-streaming created
service/video-streaming created


## List Deployments, Pods and Services

- The kubectl command:
  - `kubectl get deployments -o wide` lists all Deployments in the kubernetes cluster.
  - `kubectl get pods -o wide` lists all Pods in the kubernetes cluster.
  - `kubectl get services -o wide` lists all Services in the kubernetes cluster.
- We see that the Deployment, with associated Pods (1 replica), and Service defined in the applied manifest have been created.
- Furhermote, we see the Load Balancer's `EXTERNAL-IP` (public IP) which is the IP address you use to access your Service externally.
  - **This is the IP address you would use to access your Service publically over the internet, e.g. http://EXTERNAL-IP**

In [50]:
!kubectl get deployments
!kubectl get pods
!kubectl get services

# Let's store the Load Balancer's EXTERNAL-IP (public IP) in a Python variable so we can use it later in this notebook
LOADBALANCER_PUBLIC_IP=!kubectl get service video-streaming -o jsonpath='{.status.loadBalancer.ingress[0].ip}'
LOADBALANCER_PUBLIC_IP=LOADBALANCER_PUBLIC_IP[0]

NAME              READY   UP-TO-DATE   AVAILABLE   AGE
video-streaming   1/1     1            1           83s
NAME                               READY   STATUS    RESTARTS   AGE
video-streaming-77fc4df669-km8bj   1/1     Running   0          83s
NAME              TYPE           CLUSTER-IP     EXTERNAL-IP    PORT(S)        AGE
kubernetes        ClusterIP      10.0.0.1       <none>         443/TCP        37m
video-streaming   LoadBalancer   10.0.245.100   20.8.225.194   80:31171/TCP   83s


## Test the Microservice Application in the Kubernetes Cluster

- Open a browser and enter the URL `http://EXTERNAL-IP/SampleVideo_1280x720_1mb.mp4`
  - This sends an HTTP GET request to the Microservice's GET route for the `/video` path.
  - Streams the video `SampleVideo_1280x720_1mb.mp4` stored in the Microservice's container's file system to the browser.

In [51]:
#!firefox http://{LOADBALANCER_PUBLIC_IP}/SampleVideo_1280x720_1mb.mp4

## Delete Deployment in the Azure Kubernets Cluster

- The kubectl command `kubectl delete -f manifests/deployment.yaml` will:
  - Delete the resources defined in the file `manifests/deployment.yaml` from the kubernetes cluster.

In [52]:
!kubectl delete -f manifests/deployment.yaml

deployment.apps "video-streaming" deleted
service "video-streaming" deleted


## List Deployments, Pods and Services

- The kubectl command:
  - `kubectl get deployments -o wide` lists all Deployments in the kubernetes cluster.
  - `kubectl get pods -o wide` lists all Pods in the kubernetes cluster.
  - `kubectl get services -o wide` lists all Services in the kubernetes cluster.
- We see that the Deployment, with associated Pods (1 replica), and Service defined in the manifest file have been deleted.

In [53]:
!kubectl get deployments
!kubectl get pods
!kubectl get services

No resources found in default namespace.


NAME                               READY   STATUS        RESTARTS   AGE
video-streaming-77fc4df669-km8bj   1/1     Terminating   0          2m13s
NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.0.0.1     <none>        443/TCP   38m


## Ensure the Kubectl Context is set to the `docker-desktop` Cluster

**Note!**

- Replace `docker-desktop` below with the name of your local (development) kubernetes cluster (mine is called `docker-desktop`).
- You can list all contexts defined in kubectls config file using the command `kubectl config get-contexts`.
- If you don't have another context in your config file:
  - Use the kubectl command `kubectl config unset current-context` instead of `kubectl config use-context docker-desktop`.

In [54]:
#!kubectl config get-contexts
!kubectl config use-context docker-desktop
#!kubectl config unset current-context
!kubectl config current-context

Switched to context "docker-desktop".
docker-desktop


## Remove Azure Kubernetes Cluster Info. from Local Kubectl Config File

- The kubectl command:
  - `kubectl config delete-cluster flixtube2025g00` deletes the cluster called `flixtube2025g00` from kubectl's config file.
  - `kubectl config delete-context flixtube2025g00` deletes the context celled `flixtube2025g00` from kubectl's config file.
  - `kubectl config delete-user flixtube2025g00` deletes the user called `flixtube2025g00` from kubectl's config file.
  - `kubectl config view` lists the contents of kubectl's config file.
- kubectl's config file is located in your host machine's file system.
    - The config file is located at `~/.kube/config` on Linux/macOs.
    - The config file is located at `%USERPROFILE%\.kube\config` on Windows.

**Note**

- Reaplce `flixtube2025g00` with the value you chose for the Terraform variable `app_name` (in the file `variables.tf`).

In [55]:
!kubectl config delete-cluster flixtube2025g00
!kubectl config delete-context flixtube2025g00
!kubectl config delete-user clusterUser_flixtube2025g00_flixtube2025g00
#!kubectl config view

deleted cluster flixtube2025g00 from C:\Users\PAGA\.kube\config
deleted context flixtube2025g00 from C:\Users\PAGA\.kube\config
deleted user clusterUser_flixtube2025g00_flixtube2025g00 from C:\Users\PAGA\.kube\config


## Terraform Destroy

- We destroy the Terraform Project as before.

- Once again it's better to run this command in a separate terminal.
  - Open a new terminal.
  - Make sure you are in the folder `workshop5/01_Azure_and_Terraform/04_kubernetes_cluster`
  - Execute the command `terraform destroy -auto-approve`

The output should look something like the below ...

```bash
azurerm_resource_group.networkwatcher: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG]
azurerm_resource_group.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_network_watcher.networkwatcher: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope]
azurerm_container_registry.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_kubernetes_cluster.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00]
azurerm_role_assignment.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/a3abce92-e84a-94c5-c9dc-a5b2074e1fde]

Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # azurerm_container_registry.main will be destroyed
  - resource "azurerm_container_registry" "main" {
      - admin_enabled                 = true -> null
      - admin_password                = (sensitive value) -> null
      - admin_username                = "flixtube2025g00" -> null
      - anonymous_pull_enabled        = false -> null
      - data_endpoint_enabled         = false -> null
      - encryption                    = [] -> null
      - export_policy_enabled         = true -> null
      - id                            = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00" -> null
      - location                      = "westeurope" -> null
      - login_server                  = "flixtube2025g00.azurecr.io" -> null
      - name                          = "flixtube2025g00" -> null
      - network_rule_bypass_option    = "AzureServices" -> null
      - network_rule_set              = [] -> null
      - public_network_access_enabled = true -> null
      - quarantine_policy_enabled     = false -> null
      - resource_group_name           = "flixtube2025g00" -> null
      - retention_policy_in_days      = 0 -> null
      - sku                           = "Basic" -> null
      - tags                          = {} -> null
      - trust_policy_enabled          = false -> null
      - zone_redundancy_enabled       = false -> null
    }

  # azurerm_kubernetes_cluster.main will be destroyed
  - resource "azurerm_kubernetes_cluster" "main" {
      - cost_analysis_enabled               = false -> null
      - current_kubernetes_version          = "1.30.6" -> null
      - dns_prefix                          = "flixtube2025g00" -> null
      - fqdn                                = "flixtube2025g00-ed2io0ms.hcp.westeurope.azmk8s.io" -> null
      - id                                  = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00" -> null
      - kube_admin_config                   = (sensitive value) -> null
      - kube_config                         = (sensitive value) -> null
      - kube_config_raw                     = (sensitive value) -> null
      - kubernetes_version                  = "1.30.6" -> null
      - local_account_disabled              = false -> null
      - location                            = "westeurope" -> null
      - name                                = "flixtube2025g00" -> null
      - node_os_upgrade_channel             = "NodeImage" -> null
      - node_resource_group                 = "MC_flixtube2025g00_flixtube2025g00_westeurope" -> null
      - node_resource_group_id              = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/MC_flixtube2025g00_flixtube2025g00_westeurope" -> null
      - oidc_issuer_enabled                 = false -> null
      - portal_fqdn                         = "flixtube2025g00-ed2io0ms.portal.hcp.westeurope.azmk8s.io" -> null
      - private_cluster_enabled             = false -> null
      - private_cluster_public_fqdn_enabled = false -> null
      - resource_group_name                 = "flixtube2025g00" -> null
      - role_based_access_control_enabled   = true -> null
      - run_command_enabled                 = true -> null
      - sku_tier                            = "Free" -> null
      - support_plan                        = "KubernetesOfficial" -> null
      - tags                                = {} -> null
      - workload_identity_enabled           = false -> null
        # (8 unchanged attributes hidden)

      - default_node_pool {
          - auto_scaling_enabled          = false -> null
          - fips_enabled                  = false -> null
          - host_encryption_enabled       = false -> null
          - kubelet_disk_type             = "OS" -> null
          - max_count                     = 0 -> null
          - max_pods                      = 250 -> null
          - min_count                     = 0 -> null
          - name                          = "default" -> null
          - node_count                    = 1 -> null
          - node_labels                   = {} -> null
          - node_public_ip_enabled        = false -> null
          - only_critical_addons_enabled  = false -> null
          - orchestrator_version          = "1.30.6" -> null
          - os_disk_size_gb               = 128 -> null
          - os_disk_type                  = "Managed" -> null
          - os_sku                        = "Ubuntu" -> null
          - scale_down_mode               = "Delete" -> null
          - tags                          = {} -> null
          - type                          = "VirtualMachineScaleSets" -> null
          - ultra_ssd_enabled             = false -> null
          - vm_size                       = "Standard_D2S_V3" -> null
          - zones                         = [] -> null
            # (10 unchanged attributes hidden)

          - upgrade_settings {
              - drain_timeout_in_minutes      = 0 -> null
              - max_surge                     = "10%" -> null
              - node_soak_duration_in_minutes = 0 -> null
            }
        }

      - identity {
          - identity_ids = [] -> null
          - principal_id = "67c08ed1-086e-45e8-ab4f-f5bc23d0e051" -> null
          - tenant_id    = "b907ed40-45b9-49d7-88d8-a4d6c026ede3" -> null
          - type         = "SystemAssigned" -> null
        }

      - kubelet_identity {
          - client_id                 = "aa9761cb-ea13-4bbc-8077-4d9a66454dcb" -> null
          - object_id                 = "96a26fbe-12fb-4c4f-878a-81f4c13e37fc" -> null
          - user_assigned_identity_id = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/MC_flixtube2025g00_flixtube2025g00_westeurope/providers/Microsoft.ManagedIdentity/userAssignedIdentities/flixtube2025g00-agentpool" -> null
        }

      - network_profile {
          - dns_service_ip      = "10.0.0.10" -> null
          - ip_versions         = [
              - "IPv4",
            ] -> null
          - load_balancer_sku   = "standard" -> null
          - network_data_plane  = "azure" -> null
          - network_plugin      = "azure" -> null
          - network_plugin_mode = "overlay" -> null
          - outbound_type       = "loadBalancer" -> null
          - pod_cidr            = "10.244.0.0/16" -> null
          - pod_cidrs           = [
              - "10.244.0.0/16",
            ] -> null
          - service_cidr        = "10.0.0.0/16" -> null
          - service_cidrs       = [
              - "10.0.0.0/16",
            ] -> null
            # (2 unchanged attributes hidden)

          - load_balancer_profile {
              - backend_pool_type           = "NodeIPConfiguration" -> null
              - effective_outbound_ips      = [
                  - "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/MC_flixtube2025g00_flixtube2025g00_westeurope/providers/Microsoft.Network/publicIPAddresses/16ddc31f-85ed-4fe0-b777-0e8effc37beb",
                ] -> null
              - idle_timeout_in_minutes     = 0 -> null
              - managed_outbound_ip_count   = 1 -> null
              - managed_outbound_ipv6_count = 0 -> null
              - outbound_ip_address_ids     = [] -> null
              - outbound_ip_prefix_ids      = [] -> null
              - outbound_ports_allocated    = 0 -> null
            }
        }
    }

  # azurerm_network_watcher.networkwatcher will be destroyed
  - resource "azurerm_network_watcher" "networkwatcher" {
      - id                  = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope" -> null
      - location            = "westeurope" -> null
      - name                = "NetworkWatcher_westeurope" -> null
      - resource_group_name = "NetworkWatcherRG" -> null
      - tags                = {} -> null
    }

  # azurerm_resource_group.main will be destroyed
  - resource "azurerm_resource_group" "main" {
      - id         = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00" -> null
      - location   = "westeurope" -> null
      - name       = "flixtube2025g00" -> null
      - tags       = {} -> null
        # (1 unchanged attribute hidden)
    }

  # azurerm_resource_group.networkwatcher will be destroyed
  - resource "azurerm_resource_group" "networkwatcher" {
      - id         = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG" -> null
      - location   = "westeurope" -> null
      - name       = "NetworkWatcherRG" -> null
      - tags       = {} -> null
        # (1 unchanged attribute hidden)
    }

  # azurerm_role_assignment.main will be destroyed
  - resource "azurerm_role_assignment" "main" {
      - id                                     = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/a3abce92-e84a-94c5-c9dc-a5b2074e1fde" -> null
      - name                                   = "a3abce92-e84a-94c5-c9dc-a5b2074e1fde" -> null
      - principal_id                           = "96a26fbe-12fb-4c4f-878a-81f4c13e37fc" -> null
      - principal_type                         = "ServicePrincipal" -> null
      - role_definition_id                     = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/providers/Microsoft.Authorization/roleDefinitions/7f951dda-4ed3-4680-a7ca-43fe172d538d" -> null
      - role_definition_name                   = "AcrPull" -> null
      - scope                                  = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00" -> null
      - skip_service_principal_aad_check       = true -> null
        # (4 unchanged attributes hidden)
    }

Plan: 0 to add, 0 to change, 6 to destroy.

Changes to Outputs:
  - AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io" -> null
  - AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value) -> null
  - AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00" -> null
azurerm_network_watcher.networkwatcher: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope]
azurerm_role_assignment.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/a3abce92-e84a-94c5-c9dc-a5b2074e1fde]
azurerm_role_assignment.main: Destruction complete after 5s
azurerm_container_registry.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_kubernetes_cluster.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00]
azurerm_network_watcher.networkwatcher: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...workWatchers/NetworkWatcher_westeurope, 10s elapsed]
azurerm_network_watcher.networkwatcher: Destruction complete after 15s
azurerm_resource_group.networkwatcher: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG]
azurerm_container_registry.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...nerRegistry/registries/flixtube2025g00, 10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 10s elapsed]
azurerm_container_registry.main: Destruction complete after 19s
azurerm_resource_group.networkwatcher: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...575c23/resourceGroups/NetworkWatcherRG, 10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 20s elapsed]
azurerm_resource_group.networkwatcher: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...575c23/resourceGroups/NetworkWatcherRG, 20s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 30s elapsed]
azurerm_resource_group.networkwatcher: Destruction complete after 22s
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m0s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m20s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m30s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m0s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m20s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m30s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m0s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m20s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m30s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 4m0s elapsed]  
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 4m10s elapsed] 
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 4m20s elapsed] 
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 4m30s elapsed] 
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 4m40s elapsed] 
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 4m50s elapsed] 
azurerm_kubernetes_cluster.main: Destruction complete after 4m52s
azurerm_resource_group.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 10s elapsed]       
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 20s elapsed]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 30s elapsed]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 40s elapsed]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00, 50s elapsed]
azurerm_resource_group.main: Destruction complete after 53s

Destroy complete! Resources: 6 destroyed.
```

In [56]:
#!terraform destroy -auto-approve

## List Azure Kubernetes Services

- The Azure CLI command `az aks list -o table` lists all Azure Kubernetes Services in Azure.
- We see that the Azure Kubernetes Service (AKS) defined in the Terraform project has been destroyed.

In [57]:
!az aks list -o table

## List Azure Container Registries

- The Azure CLI command `az acr list -o table` lists all Container Registries in Azure.
- We see that the Container Registry defined in the Terraform project has been destroyed.

In [58]:
!az acr list -o table

## List Azure Resource Groups

- The Azure CLI command `az group list -o table` lists all Resource Groups in Azure.
- We see that the two Resource Groups defined in the Terraform project has been destroyed.

In [59]:
!az group list -o table